In [1]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.impute import SimpleImputer
import joblib
import os

In [2]:
landmark_path = "D:\\IDE Repo\\Dl-net\\data\\casme2-preprocessed-v2\\face_landmarks.csv"   
train_path = "D:\\IDE Repo\\Dl-net\\data\\casme2-preprocessed-v2\\train.csv" 
SAVE_DIR = "processed"
os.makedirs(SAVE_DIR, exist_ok=True)

In [3]:
landmarks_df = pd.read_csv(landmark_path)
train_df = pd.read_csv(train_path)
print("=" * 60)
print("Landmarks Dataset")
print(landmarks_df.shape)
print("=" * 60)
print("Train Dataset")
print(train_df.shape)
landmarks_df.head()


Landmarks Dataset
(500, 1436)
Train Dataset
(13615, 14)


,filepath,landmarks_detected,x_0,y_0,z_0,x_1,y_1,z_1,x_2,y_2,...,z_474,x_475,y_475,z_475,x_476,y_476,z_476,x_477,y_477,z_477
0,./data/casme2-preprocessed-v2\CASME2 Preproces...,478,0.514768,0.722219,-0.136648,0.523118,0.576818,-0.294961,0.517824,0.617790,...,0.025961,0.768488,0.236143,0.025960,0.713368,0.271990,0.025894,0.768613,0.307834,0.025907
1,./data/casme2-preprocessed-v2\CASME2 Preproces...,478,0.503075,0.703677,-0.148351,0.493394,0.541940,-0.285421,0.497619,0.590272,...,0.017879,0.768665,0.234639,0.017880,0.715009,0.270061,0.017815,0.770119,0.305565,0.017829
2,./data/casme2-preprocessed-v2\CASME2 Preproces...,478,0.500352,0.686904,-0.110848,0.499011,0.567026,-0.270116,0.498298,0.594931,...,0.000029,0.764490,0.239815,0.000024,0.711010,0.275667,-0.000036,0.763916,0.310359,-0.000028
3,./data/casme2-preprocessed-v2\CASME2 Preproces...,478,0.501748,0.691448,-0.144714,0.504308,0.553497,-0.295761,0.501742,0.600460,...,0.034370,0.796956,0.252263,0.034371,0.740294,0.285858,0.034301,0.794857,0.321623,0.034317
4,./data/casme2-preprocessed-v2\CASME2 Preproces...,478,0.485267,0.679308,-0.135492,0.496794,0.537978,-0.274870,0.493410,0.576039,...,0.013004,0.766617,0.254572,0.013000,0.715819,0.286032,0.012945,0.764681,0.318232,0.012957


In [4]:
train_df.head()

,filepath,extension,width,height,resolution,mode,aspect_ratio,brightness,contrast,sharpness,edge_density,entropy,noise,class
0,./data/casme2-preprocessed-v2/CASME2 Preproces...,.jpg,259,308,79772,RGB,0.841,90.150930,36.217449,59.012542,0.010492,7.070181,92.900850,surprise
1,./data/casme2-preprocessed-v2/CASME2 Preproces...,.jpg,259,313,81067,RGB,0.827,113.415163,37.030567,34.598467,0.005847,7.131897,100.353917,disgust
2,./data/casme2-preprocessed-v2/CASME2 Preproces...,.jpg,263,315,82845,RGB,0.835,84.742664,29.501137,28.869772,0.008896,6.793766,88.410561,disgust
3,./data/casme2-preprocessed-v2/CASME2 Preproces...,.jpg,284,321,91164,RGB,0.885,95.845882,40.332515,39.903049,0.010969,6.938547,93.265230,happy
4,./data/casme2-preprocessed-v2/CASME2 Preproces...,.jpg,256,312,79872,RGB,0.821,114.856971,36.516864,44.872642,0.008075,7.082653,103.252668,disgust


In [5]:
print('landmarks filepath sample:')
print(landmarks_df['filepath'].head(10).to_string(index=False))
print('\ntrain filepath sample:')
print(train_df['filepath'].head(10).to_string(index=False))
print('\nlandmarks basename sample:')
print(landmarks_df['filepath'].astype(str).str.replace('\\\\', '/', regex=False).str.split('/').str[-1].head(10).to_string(index=False))
print('\ntrain basename sample:')
print(train_df['filepath'].astype(str).str.replace('\\\\', '/', regex=False).str.split('/').str[-1].head(10).to_string(index=False))

landmarks filepath sample:
./data/casme2-preprocessed-v2\CASME2 Preprocess...
./data/casme2-preprocessed-v2\CASME2 Preprocess...
./data/casme2-preprocessed-v2\CASME2 Preprocess...
./data/casme2-preprocessed-v2\CASME2 Preprocess...
./data/casme2-preprocessed-v2\CASME2 Preprocess...
./data/casme2-preprocessed-v2\CASME2 Preprocess...
./data/casme2-preprocessed-v2\CASME2 Preprocess...
./data/casme2-preprocessed-v2\CASME2 Preprocess...
./data/casme2-preprocessed-v2\CASME2 Preprocess...
./data/casme2-preprocessed-v2\CASME2 Preprocess...

train filepath sample:
./data/casme2-preprocessed-v2/CASME2 Preprocess...
./data/casme2-preprocessed-v2/CASME2 Preprocess...
./data/casme2-preprocessed-v2/CASME2 Preprocess...
./data/casme2-preprocessed-v2/CASME2 Preprocess...
./data/casme2-preprocessed-v2/CASME2 Preprocess...
./data/casme2-preprocessed-v2/CASME2 Preprocess...
./data/casme2-preprocessed-v2/CASME2 Preprocess...
./data/casme2-preprocessed-v2/CASME2 Preprocess...
./data/casme2-preprocessed-v2/C

In [6]:
# Data from train.csv
label_column = "class"

# Landmark columns from face_landmarks.csv
x_cols = sorted(
    [c for c in landmarks_df.columns if c.startswith("x_")],
    key=lambda x: int(x.split("_")[1]),
)
y_cols = sorted(
    [c for c in landmarks_df.columns if c.startswith("y_")],
    key=lambda x: int(x.split("_")[1]),
)
z_cols = sorted(
    [c for c in landmarks_df.columns if c.startswith("z_")],
    key=lambda x: int(x.split("_")[1]),
)

print("Number of X landmarks:", len(x_cols))
print("Number of Y landmarks:", len(y_cols))
print("Number of Z landmarks:", len(z_cols))

assert len(x_cols) == len(y_cols) == len(z_cols), "Landmark columns do not match."
num_landmarks = len(x_cols)

print("Detected Landmarks:", num_landmarks)

Number of X landmarks: 478
Number of Y landmarks: 478
Number of Z landmarks: 478
Detected Landmarks: 478


In [7]:
# ----------------------------------------------------------
# Create Feature Matrix from multiple datasets (CASME2, CK+, SAMM)
# Uses x,y coordinates only to match model in_channels=2
# ----------------------------------------------------------

datasets = [
    {
        "name": "casme2",
        "landmark": "D:\\IDE Repo\\Dl-net\\data\\casme2-preprocessed-v2\\face_landmarks.csv",
        "train": "D:\\IDE Repo\\Dl-net\\data\\casme2-preprocessed-v2\\train.csv",
    },
    {
        "name": "ckplus",
        "landmark": "D:\\IDE Repo\\Dl-net\\data\\ckplusferdata\\face_landmarks.csv",
        "train": "D:\\IDE Repo\\Dl-net\\data\\ckplusferdata\\train.csv",
    },
    {
        "name": "samm",
        "landmark": "D:\\IDE Repo\\Dl-net\\data\\sammassamexpression\\face_landmarks.csv",
        "train": "D:\\IDE Repo\\Dl-net\\data\\sammassamexpression\\train.csv",
    },
]

merged_list = []
label_column = "class"

for ds in datasets:
    if not (os.path.exists(ds["landmark"]) and os.path.exists(ds["train"])):
        print(f"Skipping {ds['name']} — files not found: {ds['landmark']} or {ds['train']}")
        continue

    ldf = pd.read_csv(ds["landmark"]).copy()
    tdf = pd.read_csv(ds["train"]).copy()

    # normalize paths and merge
    ldf["filepath_norm"] = ldf["filepath"].astype(str).str.strip().str.replace("\\", "/", regex=False)
    tdf["filepath_norm"] = tdf["filepath"].astype(str).str.strip().str.replace("\\", "/", regex=False)
    
    print(f"{ds['name']} - Sample landmark paths:", ldf["filepath_norm"].head(2).tolist())
    print(f"{ds['name']} - Sample train paths:", tdf["filepath_norm"].head(2).tolist())

    # determine x/y cols in this landmark file
    x_cols_local = sorted([c for c in ldf.columns if c.startswith("x_")], key=lambda x: int(x.split("_")[1]))
    y_cols_local = sorted([c for c in ldf.columns if c.startswith("y_")], key=lambda x: int(x.split("_")[1]))

    if len(x_cols_local) == 0 or len(y_cols_local) == 0:
        print(f"No x/y landmarks found for {ds['name']}, skipping")
        continue

    # select only x and y columns
    use_cols = ["filepath_norm"] + x_cols_local + y_cols_local
    ldf_sel = ldf[use_cols]

    merged = ldf_sel.merge(tdf[["filepath_norm", label_column]], on="filepath_norm", how="inner")
    print(f"{ds['name']} merged:", merged.shape)

    if merged.empty:
        print(f"Warning: no rows matched for {ds['name']}")
        continue

    merged_list.append(merged)

if not merged_list:
    raise RuntimeError("No dataset merged — check that data files exist and filepath columns match.")

# concatenate merged datasets
merged_all = pd.concat(merged_list, ignore_index=True)
print("Combined merged dataset:", merged_all.shape)

# Determine common landmark indices (assumes same numbering scheme)
x_cols = sorted([c for c in merged_all.columns if c.startswith("x_")], key=lambda x: int(x.split("_")[1]))
y_cols = sorted([c for c in merged_all.columns if c.startswith("y_")], key=lambda x: int(x.split("_")[1]))

# Keep the minimum length in case one dataset had fewer landmarks
min_len = min(len(x_cols), len(y_cols))
if min_len == 0:
    raise RuntimeError("No x/y landmark columns found in combined data.")

x_cols = x_cols[:min_len]
y_cols = y_cols[:min_len]

print("Using landmarks count:", min_len)

features = []
for i in range(min_len):
    features.append(x_cols[i])
    features.append(y_cols[i])

# Build X and y
X = merged_all[features]
encoder = LabelEncoder()
y = encoder.fit_transform(merged_all[label_column].astype(str).values.ravel())

print("Feature Matrix:", X.shape)
print("Label Vector:", y.shape)

print("Classes:")
for i, c in enumerate(encoder.classes_):
    print(i, "->", c)

# Handle missing values
imputer = SimpleImputer(strategy="median")
X = pd.DataFrame(imputer.fit_transform(X), columns=features)

# Save preprocessing artifacts
os.makedirs(SAVE_DIR, exist_ok=True)
joblib.dump(encoder, os.path.join(SAVE_DIR, "label_encoder.joblib"))
joblib.dump(imputer, os.path.join(SAVE_DIR, "imputer.joblib"))
joblib.dump(features, os.path.join(SAVE_DIR, "feature_columns.joblib"))

print("Saved preprocessing artifacts to:", SAVE_DIR)

casme2 - Sample landmark paths: ['./data/casme2-preprocessed-v2/CASME2 Preprocessed v2/others/reg_img65 (24).jpg', './data/casme2-preprocessed-v2/CASME2 Preprocessed v2/others/reg_img135 (14).jpg']
casme2 - Sample train paths: ['./data/casme2-preprocessed-v2/CASME2 Preprocessed v2/surprise/reg_img44 (4).jpg', './data/casme2-preprocessed-v2/CASME2 Preprocessed v2/disgust/reg_img70 (18).jpg']
casme2 merged: (393, 958)
ckplus - Sample landmark paths: ['./data/ckplusferdata/train/neutral/Training_46272799.jpg', './data/ckplusferdata/train/sad/Training_8856969.jpg']
ckplus - Sample train paths: ['./data/ckplusferdata/train/angry/Training_23412616.jpg', './data/ckplusferdata/train/fear/Training_9013309.jpg']
ckplus merged: (373, 958)
samm - Sample landmark paths: ['./data/sammassamexpression/expression_data/sad/image0020786_face.jpg', './data/sammassamexpression/expression_data/neutral/ffhq_2900_face.jpg']
samm - Sample train paths: ['./data/sammassamexpression/expression_data/happy/ffhq_401

In [8]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y,
)

print("Train set:", X_train.shape, y_train.shape)
print("Test set:", X_test.shape, y_test.shape)


Train set: (928, 956) (928,)
Test set: (233, 956) (233,)


In [9]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch_geometric.data import Data
from torch_geometric.loader import DataLoader

from torch_geometric.nn import (
    GCNConv,
    ChebConv,
    TopKPooling,
    global_mean_pool,
    global_max_pool,
)


# -------------------------------------------------------
# Displacement-based graph construction
# -------------------------------------------------------
def build_micro_expression_graph(
    onset_landmarks,
    apex_landmarks,
    edge_index,
    y=None,
    normalize_displacement=True,
):
    """
    Builds a single PyG `Data` object for one micro-expression sample,
    encoding onset->apex landmark displacement — the actual signal that
    defines a micro-expression — rather than static apex coordinates alone.

    Args:
        onset_landmarks: (N, 2) tensor of (x, y) landmark coords at onset frame.
        apex_landmarks:  (N, 2) tensor of (x, y) landmark coords at apex frame.
        edge_index: (2, E) tensor defining graph topology (e.g. facial mesh
            adjacency, or a k-NN graph built once from a mean face template).
        y: optional label tensor, e.g. torch.tensor([class_idx]).
        normalize_displacement: if True, scales displacement by the face's
            own inter-ocular distance (landmarks 0/1 assumed as eye corners)
            so magnitude is comparable across subjects/face sizes. Adjust the
            landmark indices to match your landmark scheme.

    Node features (per landmark), shape (N, 5):
        [apex_x, apex_y, disp_x, disp_y, disp_magnitude]

    Edge features (per edge), shape (E, 1):
        mean displacement magnitude of the edge's two endpoint nodes —
        edges connecting landmarks that moved more carry a larger weight,
        so the conv layers attend more to the muscles that actually moved.
    """
    onset_landmarks = onset_landmarks.float()
    apex_landmarks = apex_landmarks.float()

    displacement = apex_landmarks - onset_landmarks  # (N, 2)

    if normalize_displacement:
        # inter-ocular distance as a scale reference; swap indices to match
        # your landmark ordering (e.g. dlib 68-pt uses 36 and 45)
        eye_l, eye_r = onset_landmarks[0], onset_landmarks[1]
        scale = torch.norm(eye_r - eye_l).clamp(min=1e-6)
        displacement = displacement / scale

    disp_magnitude = torch.norm(displacement, dim=1, keepdim=True)  # (N, 1)

    x = torch.cat([apex_landmarks, displacement, disp_magnitude], dim=1)  # (N, 5)

    src, dst = edge_index[0], edge_index[1]
    edge_disp = (disp_magnitude[src] + disp_magnitude[dst]) / 2.0  # (E, 1)
    edge_attr = edge_disp.view(-1)

    data = Data(x=x, edge_index=edge_index, edge_attr=edge_attr)
    if y is not None:
        data.y = y
    return data


# -------------------------------------------------------
# Spatial Branch (SPAGNN)
# -------------------------------------------------------
class SPAGNN(nn.Module):
    """
    Spatial branch operating directly on landmark graph topology.
    Improvements over baseline:
      - BatchNorm after each conv for training stability
      - Residual connection across the two conv blocks
      - Edge-weight support (e.g. inverse landmark distance)
      - Mean+Max pooling concat instead of mean-only (richer graph summary)
    """

    def __init__(self, in_channels, hidden_channels=64, dropout=0.3):
        super().__init__()

        self.conv1 = GCNConv(in_channels, hidden_channels)
        self.bn1 = nn.BatchNorm1d(hidden_channels)
        self.pool1 = TopKPooling(hidden_channels, ratio=0.8)

        self.conv2 = GCNConv(hidden_channels, hidden_channels)
        self.bn2 = nn.BatchNorm1d(hidden_channels)
        self.pool2 = TopKPooling(hidden_channels, ratio=0.8)

        # projects residual from block1 dim -> block2 dim (identity here,
        # kept explicit in case hidden dims ever differ per block)
        self.res_proj = nn.Linear(hidden_channels, hidden_channels)

        self.dropout = nn.Dropout(dropout)
        # mean + max pooled features concatenated -> project back down
        self.fc = nn.Linear(hidden_channels * 2, hidden_channels)

    def forward(self, data):
        x = data.x
        edge_index = data.edge_index
        batch = data.batch
        edge_weight = getattr(data, "edge_attr", None)
        if edge_weight is not None and edge_weight.dim() > 1:
            edge_weight = edge_weight.view(-1)

        x = F.relu(self.bn1(self.conv1(x, edge_index, edge_weight)))
        x, edge_index, edge_weight, batch, _, _ = self.pool1(
            x, edge_index, edge_weight, batch=batch
        )
        x = self.dropout(x)

        residual = self.res_proj(x)
        x = F.relu(self.bn2(self.conv2(x, edge_index, edge_weight)))
        x = x + residual  # residual connection

        x, edge_index, edge_weight, batch, _, _ = self.pool2(
            x, edge_index, edge_weight, batch=batch
        )
        x = self.dropout(x)

        x_mean = global_mean_pool(x, batch)
        x_max = global_max_pool(x, batch)
        x = torch.cat([x_mean, x_max], dim=1)

        x = F.relu(self.fc(x))
        return x


# -------------------------------------------------------
# Spectral Branch (SPEGNN)
# -------------------------------------------------------
class SPEGNN(nn.Module):
    """
    Spectral branch using Chebyshev graph convolutions.
    Improvements over baseline:
      - Two ChebConv layers instead of one (baseline had a single-layer
        spectral branch, which is shallow relative to the spatial branch)
      - Correct per-graph lambda_max estimation via `batch=` kwarg
        (critical bug fix: without this, batched graphs share a single
        lambda_max estimate, corrupting the spectral normalization)
      - BatchNorm + residual, mirroring the spatial branch
      - Edge-weight support
    """

    def __init__(self, in_channels, hidden_channels=64, K=3, dropout=0.3):
        super().__init__()

        self.conv1 = ChebConv(in_channels, hidden_channels, K=K)
        self.bn1 = nn.BatchNorm1d(hidden_channels)

        self.conv2 = ChebConv(hidden_channels, hidden_channels, K=K)
        self.bn2 = nn.BatchNorm1d(hidden_channels)

        self.res_proj = nn.Linear(hidden_channels, hidden_channels)
        self.dropout = nn.Dropout(dropout)
        self.fc = nn.Linear(hidden_channels * 2, hidden_channels)

    def forward(self, data):
        edge_weight = getattr(data, "edge_attr", None)
        if edge_weight is not None and edge_weight.dim() > 1:
            edge_weight = edge_weight.view(-1)

        x = F.relu(
            self.bn1(
                self.conv1(data.x, data.edge_index, edge_weight, batch=data.batch)
            )
        )
        x = self.dropout(x)

        residual = self.res_proj(x)
        x = F.relu(
            self.bn2(
                self.conv2(x, data.edge_index, edge_weight, batch=data.batch)
            )
        )
        x = x + residual

        x_mean = global_mean_pool(x, data.batch)
        x_max = global_max_pool(x, data.batch)
        x = torch.cat([x_mean, x_max], dim=1)

        x = F.relu(self.fc(x))
        return x


# -------------------------------------------------------
# Attention-based fusion of the two branches
# -------------------------------------------------------
class BranchAttentionFusion(nn.Module):
    """
    Learns a soft weight for each branch's contribution per sample,
    instead of a fixed concatenation. Often outperforms naive concat
    when one branch is more informative for a given sample/class.
    """

    def __init__(self, hidden_channels):
        super().__init__()
        self.attn = nn.Sequential(
            nn.Linear(hidden_channels * 2, hidden_channels),
            nn.Tanh(),
            nn.Linear(hidden_channels, 2),
        )

    def forward(self, spatial_feat, spectral_feat):
        combined = torch.cat([spatial_feat, spectral_feat], dim=1)
        weights = F.softmax(self.attn(combined), dim=1)  # (B, 2)
        w_spatial = weights[:, 0].unsqueeze(1)
        w_spectral = weights[:, 1].unsqueeze(1)
        fused = torch.cat(
            [w_spatial * spatial_feat, w_spectral * spectral_feat], dim=1
        )
        return fused, weights


# -------------------------------------------------------
# Complete SSGNN
# -------------------------------------------------------
class SSGNN(nn.Module):
    """
    Improvements over baseline:
      - Deeper, normalized, residual branches (see above)
      - Attention-based branch fusion instead of plain concat
      - Label smoothing-friendly classifier head with an extra
        hidden layer + BatchNorm for better generalization on small
        datasets like CASME II
    """

    def __init__(self, in_channels, hidden_channels, num_classes, dropout=0.5):
        super().__init__()

        self.spatial = SPAGNN(in_channels, hidden_channels)
        self.spectral = SPEGNN(in_channels, hidden_channels)

        self.fusion = BranchAttentionFusion(hidden_channels)

        self.classifier = nn.Sequential(
            nn.Linear(hidden_channels * 2, hidden_channels),
            nn.BatchNorm1d(hidden_channels),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(hidden_channels, hidden_channels // 2),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(hidden_channels // 2, num_classes),
        )

    def forward(self, data, return_attn=False):
        spatial_feat = self.spatial(data)
        spectral_feat = self.spectral(data)

        fused, attn_weights = self.fusion(spatial_feat, spectral_feat)
        out = self.classifier(fused)

        if return_attn:
            return out, attn_weights
        return out


c:\Users\Sad Bin Siddique\.conda\envs\ml\Lib\site-packages\torch_geometric\__init__.py:4: UserWarning: An issue occurred while importing 'torch-scatter'. Disabling its usage. Stacktrace: Could not load this library: C:\Users\Sad Bin Siddique\.conda\envs\ml\Lib\site-packages\torch_scatter\_version_cuda.pyd
  import torch_geometric.typing
c:\Users\Sad Bin Siddique\.conda\envs\ml\Lib\site-packages\torch_geometric\__init__.py:4: UserWarning: An issue occurred while importing 'torch-sparse'. Disabling its usage. Stacktrace: Could not load this library: C:\Users\Sad Bin Siddique\.conda\envs\ml\Lib\site-packages\torch_sparse\_version_cuda.pyd
  import torch_geometric.typing


In [10]:
import torch

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# in_channels=2: each landmark node is a static (x, y) coordinate
# (no onset/apex displacement available in this single-frame pipeline)
model = SSGNN(
    in_channels=2,
    hidden_channels=128,
    num_classes=len(encoder.classes_),
).to(device)

print(device)
print(model)


cuda
SSGNN(
  (spatial): SPAGNN(
    (conv1): GCNConv(2, 128)
    (bn1): BatchNorm1d(128, eps=1e-05, momentum=0.1, affine=True, bias=True, track_running_stats=True)
    (pool1): TopKPooling(128, ratio=0.8, multiplier=1.0)
    (conv2): GCNConv(128, 128)
    (bn2): BatchNorm1d(128, eps=1e-05, momentum=0.1, affine=True, bias=True, track_running_stats=True)
    (pool2): TopKPooling(128, ratio=0.8, multiplier=1.0)
    (res_proj): Linear(in_features=128, out_features=128, bias=True)
    (dropout): Dropout(p=0.3, inplace=False)
    (fc): Linear(in_features=256, out_features=128, bias=True)
  )
  (spectral): SPEGNN(
    (conv1): ChebConv(2, 128, K=3, normalization=sym)
    (bn1): BatchNorm1d(128, eps=1e-05, momentum=0.1, affine=True, bias=True, track_running_stats=True)
    (conv2): ChebConv(128, 128, K=3, normalization=sym)
    (bn2): BatchNorm1d(128, eps=1e-05, momentum=0.1, affine=True, bias=True, track_running_stats=True)
    (res_proj): Linear(in_features=128, out_features=128, bias=True)

In [11]:
from sklearn.utils.class_weight import compute_class_weight

weights = compute_class_weight('balanced', classes=np.unique(y_train), y=y_train)
criterion = torch.nn.CrossEntropyLoss(weight=torch.tensor(weights, dtype=torch.float).to(device))



# ==========================================================
# Optimizer
# ==========================================================
optimizer = torch.optim.AdamW(
    model.parameters(),
    lr=1e-4,
    weight_decay=1e-4
)

# ==========================================================
# Learning Rate Scheduler (Optional but Recommended)
# ==========================================================
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
    optimizer,
    mode='max',          # Monitor validation accuracy
    factor=0.5,          # LR = LR × 0.5
    patience=3,
    threshold=1e-4,
    min_lr=1e-7
)


In [12]:
from tqdm import tqdm

def train(model, loader, optimizer, criterion):
    model.train()

    total_loss = 0
    correct = 0
    total = 0

    for data in tqdm(loader):

        data = data.to(device)

        optimizer.zero_grad()

        out = model(data)

        loss = criterion(out, data.y)

        loss.backward()
        
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)

        optimizer.step()

        total_loss += loss.item()

        pred = out.argmax(dim=1)

        correct += (pred == data.y).sum().item()

        total += data.y.size(0)

    loss = total_loss / len(loader)
    acc = correct / total

    return loss, acc

In [13]:
@torch.no_grad()
def evaluate(model, loader, criterion):

    model.eval()

    total_loss = 0
    correct = 0
    total = 0

    for data in loader:

        data = data.to(device)

        out = model(data)

        loss = criterion(out, data.y)

        total_loss += loss.item()

        pred = out.argmax(dim=1)

        correct += (pred == data.y).sum().item()

        total += data.y.size(0)

    loss = total_loss / len(loader)
    acc = correct / total

    return loss, acc

In [14]:
import torch
from torch_geometric.data import Data
from torch_geometric.loader import DataLoader

# ============================================================
# Real MediaPipe FaceMesh anatomical connections (not a naive chain)
# Source: mediapipe/python/solutions/face_mesh_connections.py
# (Apache 2.0, google-ai-edge/mediapipe). Indices are mediapipe's
# canonical 468-point numbering; only edges where BOTH endpoints are
# < num_landmarks are kept, since our merged CSVs may only retain the
# first `min_len` landmark columns.
# ============================================================
_FACEMESH_LIPS = [
    (61, 146), (146, 91), (91, 181), (181, 84), (84, 17),
    (17, 314), (314, 405), (405, 321), (321, 375),
    (375, 291), (61, 185), (185, 40), (40, 39), (39, 37),
    (37, 0), (0, 267), (267, 269), (269, 270), (270, 409), (409, 291),
    (78, 95), (95, 88), (88, 178), (178, 87), (87, 14),
    (14, 317), (317, 402), (402, 318), (318, 324),
    (324, 308), (78, 191), (191, 80), (80, 81), (81, 82),
    (82, 13), (13, 312), (312, 311), (311, 310),
    (310, 415), (415, 308),
]
_FACEMESH_LEFT_EYE = [
    (263, 249), (249, 390), (390, 373), (373, 374),
    (374, 380), (380, 381), (381, 382), (382, 362),
    (263, 466), (466, 388), (388, 387), (387, 386),
    (386, 385), (385, 384), (384, 398), (398, 362),
]
_FACEMESH_LEFT_EYEBROW = [
    (276, 283), (283, 282), (282, 295),
    (295, 285), (300, 293), (293, 334),
    (334, 296), (296, 336),
]
_FACEMESH_RIGHT_EYE = [
    (33, 7), (7, 163), (163, 144), (144, 145),
    (145, 153), (153, 154), (154, 155), (155, 133),
    (33, 246), (246, 161), (161, 160), (160, 159),
    (159, 158), (158, 157), (157, 173), (173, 133),
]
_FACEMESH_RIGHT_EYEBROW = [
    (46, 53), (53, 52), (52, 65), (65, 55),
    (70, 63), (63, 105), (105, 66), (66, 107),
]
_FACEMESH_FACE_OVAL = [
    (10, 338), (338, 297), (297, 332), (332, 284),
    (284, 251), (251, 389), (389, 356), (356, 454),
    (454, 323), (323, 361), (361, 288), (288, 397),
    (397, 365), (365, 379), (379, 378), (378, 400),
    (400, 377), (377, 152), (152, 148), (148, 176),
    (176, 149), (149, 150), (150, 136), (136, 172),
    (172, 58), (58, 132), (132, 93), (93, 234),
    (234, 127), (127, 162), (162, 21), (21, 54),
    (54, 103), (103, 67), (67, 109), (109, 10),
]
_FACEMESH_NOSE = [
    (168, 6), (6, 197), (197, 195), (195, 5),
    (5, 4), (4, 1), (1, 19), (19, 94), (94, 2), (98, 97),
    (97, 2), (2, 326), (326, 327), (327, 294),
    (294, 278), (278, 344), (344, 440), (440, 275),
    (275, 4), (4, 45), (45, 220), (220, 115), (115, 48),
    (48, 64), (64, 98),
]

_ALL_MEDIAPIPE_EDGES = (
    _FACEMESH_LIPS + _FACEMESH_LEFT_EYE + _FACEMESH_LEFT_EYEBROW
    + _FACEMESH_RIGHT_EYE + _FACEMESH_RIGHT_EYEBROW + _FACEMESH_FACE_OVAL
    + _FACEMESH_NOSE
)


def build_mediapipe_edge_index(num_landmarks, bidirectional=True):
    """
    Builds a (2, E) edge_index from mediapipe's real facial contour
    topology (eyes, eyebrows, lips, face oval, nose), restricted to
    edges whose endpoints both fall within `num_landmarks`.

    Replaces the previous naive chain (`edges.append([i, i+1])`), which
    connected landmarks purely by column index order and produced
    anatomically meaningless edges (e.g. jaw-point 16 wired directly to
    eyebrow-point 17).
    """
    edges = [(a, b) for (a, b) in _ALL_MEDIAPIPE_EDGES if a < num_landmarks and b < num_landmarks]
    if len(edges) < num_landmarks // 4:
        print(
            f"Warning: only {len(edges)} anatomical edges fit within "
            f"{num_landmarks} landmarks. Your landmark CSV may use a "
            f"different index ordering than mediapipe's canonical scheme, "
            f"or num_landmarks is unusually small — double check before training."
        )
    edge_index = torch.tensor(edges, dtype=torch.long).t().contiguous()
    if bidirectional:
        edge_index = torch.cat([edge_index, edge_index.flip(0)], dim=1)
    return edge_index


edge_index = build_mediapipe_edge_index(min_len)
print("Edge index shape:", edge_index.shape, "| landmarks used:", min_len)


def create_graph_dataset(X, y, edge_index, in_channels=2, normalize=True):
    """
    Converts flattened landmark feature rows into PyG graphs.

    normalize=True centers each face's landmarks on their own centroid
    and scales by their own coordinate std. This matters because we're
    merging CASME2/CK+/SAMM, which have different image resolutions and
    face crop sizes — without this, the model partly learns to
    discriminate by image scale/position rather than expression.
    """
    if hasattr(X, "values"):
        X_np = X.values
    else:
        X_np = np.asarray(X)

    dataset = []

    num_features = X_np.shape[1]
    if num_features % in_channels != 0:
        raise ValueError(f"Number of features ({num_features}) is not divisible by in_channels={in_channels}.")

    num_nodes = num_features // in_channels

    for i in range(len(X_np)):
        row = X_np[i]
        coords = row.reshape(num_nodes, in_channels).astype(np.float32)

        if normalize:
            centroid = coords.mean(axis=0, keepdims=True)
            coords = coords - centroid
            scale = coords.std() + 1e-6
            coords = coords / scale

        x = torch.tensor(coords, dtype=torch.float)

        label = torch.tensor(
            int(y[i]),
            dtype=torch.long,
        )

        graph = Data(
            x=x,
            edge_index=edge_index,
            y=label
        )

        dataset.append(graph)

    return dataset


train_dataset = create_graph_dataset(
    X_train,
    y_train,
    edge_index,
    in_channels=2,
)

test_dataset = create_graph_dataset(
    X_test,
    y_test,
    edge_index,
    in_channels=2,
)

# Create PyG DataLoaders
batch_size = 32
train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False)

print("Datasets -> train:, test:", len(train_dataset), len(test_dataset))
print("DataLoaders -> train batches:, test batches:", len(train_loader), len(test_loader))


Edge index shape: torch.Size([2, 298]) | landmarks used: 478
Datasets -> train:, test: 928 233
DataLoaders -> train batches:, test batches: 29 8


In [15]:
X_train.shape, y_train.shape, X_test.shape, y_test.shape


((928, 956), (928,), (233, 956), (233,))

In [16]:
EPOCHS = 100

best_acc = 0

train_losses = []
test_losses = []

train_accs = []
test_accs = []

for epoch in range(1, EPOCHS + 1):

    train_loss, train_acc = train(
        model,
        train_loader,
        optimizer,
        criterion
    )

    test_loss, test_acc = evaluate(
        model,
        test_loader,
        criterion
    )
    
    scheduler.step(test_acc)  # mode='max' in the scheduler -> must feed a metric that improves upward
    
    current_lr = optimizer.param_groups[0]['lr']
    

    train_losses.append(train_loss)
    test_losses.append(test_loss)

    train_accs.append(train_acc)
    test_accs.append(test_acc)

    if test_acc > best_acc:

        best_acc = test_acc

        torch.save(model.state_dict(), "best_ssgnn.pth")

    print(
        f"Epoch {epoch:03d} | "
        f"Train Loss={train_loss:.4f} Train Acc={train_acc:.4f} | "
        f"Test Loss={test_loss:.4f} Test Acc={test_acc:.4f} | "
        f"LR={current_lr:.6f}"
    )

print("\nTraining Finished!")
print("Best Accuracy:", best_acc)

  0%|          | 0/29 [00:00<?, ?it/s]c:\Users\Sad Bin Siddique\.conda\envs\ml\Lib\site-packages\torch_geometric\utils\_scatter.py:91: UserWarning: The usage of `scatter(reduce='max')` can be accelerated via the 'torch-scatter' package, but it was not found
  warnings.warn(
100%|██████████| 29/29 [00:01<00:00, 17.70it/s]


Epoch 001 | Train Loss=1.9941 Train Acc=0.1196 | Test Loss=1.9565 Test Acc=0.0858 | LR=0.000100


100%|██████████| 29/29 [00:00<00:00, 37.17it/s]


Epoch 002 | Train Loss=2.0041 Train Acc=0.1153 | Test Loss=1.9697 Test Acc=0.0858 | LR=0.000100


100%|██████████| 29/29 [00:00<00:00, 41.21it/s]


...

Epoch 098 | Train Loss=1.9150 Train Acc=0.1929 | Test Loss=1.9431 Test Acc=0.0987 | LR=0.000000


100%|██████████| 29/29 [00:00<00:00, 37.47it/s]


Epoch 099 | Train Loss=1.9137 Train Acc=0.1767 | Test Loss=1.9460 Test Acc=0.0987 | LR=0.000000


100%|██████████| 29/29 [00:00<00:00, 37.07it/s]


Epoch 100 | Train Loss=1.9168 Train Acc=0.1681 | Test Loss=1.9450 Test Acc=0.0944 | LR=0.000000

Training Finished!
Best Accuracy: 0.1759656652360515
